In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install neo4j langchain langchain-openai langchain-neo4j langchain-text-splitters langchain-huggingface python-dotenv -q langchain_anthropic

In [ ]:
import os
import argparse
from typing import List, Dict
import pprint
import json

from neo4j import GraphDatabase

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_neo4j import Neo4jVector
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import Markdown, display
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import JsonOutputParser


#for ingest batch data with cache
import pickle
import hashlib
from google.colab import drive

# Neo4j Database Setup

In [ ]:
"""Initializes and returns the Neo4j driver."""

def get_neo4j_driver():
    uri = os.getenv("NEO4J_URI")
    user = os.getenv("NEO4J_USERNAME")
    password = os.getenv("NEO4J_PASSWORD")
    driver = GraphDatabase.driver(uri, auth=(user, password))
    print("driver initialized")
    return driver

In [ ]:
def setup_neo4j_constraints(driver):
    """Adds unique constraints to the Neo4j database to prevent duplicate nodes."""

    database = os.getenv("NEO4J_DATABASE", "neo4j")
    # database = "c7a779b6"
    with driver.session(database=database) as session:
        session.run("CREATE CONSTRAINT city_unique IF NOT EXISTS FOR (c:City) REQUIRE c.city_name IS UNIQUE")
        session.run("CREATE CONSTRAINT meeting_unique IF NOT EXISTS FOR (m:Meeting) REQUIRE m.meeting_id IS UNIQUE")
        session.run("CREATE CONSTRAINT item_unique IF NOT EXISTS FOR (i:Item) REQUIRE i.item_id IS UNIQUE")
        session.run("CREATE CONSTRAINT itemtype_unique IF NOT EXISTS FOR (it:ItemType) REQUIRE it.type_name IS UNIQUE")
    print("Neo4j constraints are set.")


In [ ]:

def create_vector_index(driver, embedding_model):
    """Creates a vector index on the TrancriptChunk nodes for fast similarity search."""

    database = os.getenv("NEO4J_DATABASE", "neo4j")

    #vector dimensions for nomic-embed-text-v1.5 is 768
    index_query = f"""
    CREATE VECTOR INDEX `transcript_chunk_index` IF NOT EXISTS
    FOR (c:TranscriptChunk) ON (c.embedding)
    OPTIONS {{ indexConfig: {{
        `vector.dimensions`: 768,
        `vector.similarity_function`: 'cosine'
    }} }}
    """
    with driver.session(database=database) as session:
        session.run(index_query)
    print("Vector index on TranscriptChunk nodes is created.")


# Data Ingestion and Embedding


In [ ]:

def ingest_data(driver, data, embedding_model, text_splitter):
    """
    Baseline ingestion pipeline for loading MeetingBank data into Neo4j.

    Iterates over each meeting sequentially and performs 10 steps per meeting as shown in the code.

    All writes use MERGE to ensure no duplicate data
    — safe to re-run if the pipeline is interrupted

    Embeddings are generated and written one meeting at a time.
    For large datasets this is slow; see ingest_data_resumable() for the
    production-grade version with batching, caching, and checkpoint recovery.
    """
    database = os.getenv("NEO4J_DATABASE", "neo4j")
    with driver.session(database=database) as session:
        for meeting_id, meeting_data in data.items():
            city_name = meeting_data.get('city', 'Unknown')
            meeting_date=meeting_data.get('meeting_date','Unknown')
            urls = meeting_data.get('URLs', {}) # Corrected: Changed 'urls' to 'URLs'
            item_info = meeting_data.get('itemInfo', {})

            # 1. Create or merge City node
            session.run("""
                MERGE (c:City {city_name: $city_name})
                """, city_name=city_name)

            # 2. Create or merge Meeting node
            session.run("""
                MERGE (m:Meeting {meeting_id: $meeting_id})
                SET
                        m.webpage_link = $webpage,
                        m.video_link = $video,
                        m.meetingdetail_link = $meetingdetail,
                        m.video_duration = $videoduration,
                        m.meeting_date = $meeting_date

                """,
                meeting_id=meeting_id,
                webpage=urls.get('Webpage', ''),
                video=urls.get('Video', ''),
                meetingdetail=urls.get('MeetingDetail', ''),
                videoduration=meeting_data.get('VideoDuration', 0),
                meeting_date = meeting_data.get('meeting_date')#to include meeting_date
            )

            # 3. Create relationship between City and Meeting
            session.run("""
                MATCH (c:City {city_name: $city_name}), (m:Meeting {meeting_id: $meeting_id})
                MERGE (m)-[:TAKES_PLACE_IN]->(c)
                """, city_name=city_name, meeting_id=meeting_id)

            # Process items in the meeting
            for item_id, item_data in item_info.items():
                item_type = item_data.get('type', 'Unknown')

                # 4. Create or match ItemType node
                session.run("""
                    MERGE (it:ItemType {type_name: $type_name})
                    """, type_name=item_type)

                # 5. Create Item node
                session.run("""
                    MERGE (i:Item {item_id: $item_id})
                    ON CREATE SET
                            i.start_time = $start_time,
                            i.end_time = $end_time,
                            i.duration = $duration,
                            i.summary = $summary
                    """,
                    item_id=item_id,
                    start_time=item_data.get('startTime', ''),
                    end_time=item_data.get('endTime', ''),
                    duration=item_data.get('duration', 0),
                    summary=item_data.get('Summary', '')
                )

                # 6. Create relationship between Item and ItemType
                session.run("""
                    MATCH (i:Item {item_id: $item_id}), (it:ItemType {type_name: $type_name})
                    MERGE (i)-[:OF_TYPE]->(it)
                    """, item_id=item_id, type_name=item_type)

                # 7. Create relationship between Meeting and Item
                session.run("""
                    MATCH (m:Meeting {meeting_id: $meeting_id}), (i:Item {item_id: $item_id})
                    MERGE (m)-[:HAS_ITEM]->(i)
                    """, meeting_id=meeting_id, item_id=item_id)

                # 8. Chunk and embed transcripts
                transcript = item_data.get('transcripts', '')
                if not transcript:
                    continue  # Skip if no transcript text is available

                chunks = text_splitter.split_text(transcript)
                embeddings = embedding_model.embed_documents(chunks)

                # For each chunk, create a TranscriptChunk node and link it to the Item
                for i, chunk in enumerate(chunks):
                    # 9. Create a node for each chunk and store the embedding
                    session.run("""
                        MERGE (c:TranscriptChunk {text: $text, embedding: $embedding, chunk_index: $chunk_index})
                    """,text=chunk, embedding=embeddings[i], chunk_index=i)

                    # 10. create relationship between chunk and item
                    session.run("""
                        MATCH (c:TranscriptChunk {text: $text, embedding: $embedding, chunk_index: $chunk_index}), (i:Item {item_id: $item_id})
                        MERGE (i)-[:HAS_CHUNK]->(c)
                    """, text=chunk, embedding=embeddings[i], chunk_index=i, item_id=item_id)

            print(f"Finished ingesting meeting: {meeting_id}")

    print("\n all meetings are succesfully ingested into Neo4j!")

In [ ]:
def ingest_data_resumable(driver, data, embedding_model, text_splitter,
                           batch_size=1000, embed_batch_size=512,
                           drive_cache_dir="/content/drive/MyDrive/meetingbank_cache"):
    """
    Resumable ingestion with Google Drive-backed cache.
    - Embeddings are generated in mini-batches and checkpointed to Drive after each batch.
    - On restart, already-embedded chunks are loaded from Drive and skipped.
    - Neo4j MERGE ensures already-ingested nodes are idempotent (safe to re-run).
    """
    # ── 0. Mount Drive ──────────────────────────────────────────────────────────
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    os.makedirs(drive_cache_dir, exist_ok=True)

    database = os.getenv("NEO4J_DATABASE", "neo4j")

    # ── 1. Collect metadata (fast, no embeddings yet) ───────────────────────────
    print("Collecting metadata and chunking transcripts...")
    all_city_data, all_meeting_data, all_meeting_city_rels = [], [], []
    all_item_type_data, all_item_data = [], []
    all_item_item_type_rels, all_meeting_item_rels = [], []
    raw_chunks = []  # [{'item_id', 'text', 'chunk_index'}]

    for meeting_id, meeting_data in data.items():
        city_name    = meeting_data.get('city', 'Unknown')
        urls         = meeting_data.get('URLs', {})
        item_info    = meeting_data.get('itemInfo', {})
        meeting_date = meeting_data.get('meeting_date', 'Unknown')

        all_city_data.append({'city_name': city_name})
        all_meeting_data.append({
            'meeting_id':   meeting_id,
            'webpage':      urls.get('Webpage', ''),
            'video':        urls.get('Video', ''),
            'meetingdetail':urls.get('MeetingDetail', ''),
            'videoduration':meeting_data.get('VideoDuration', 0),
            'meeting_date': meeting_date
        })
        all_meeting_city_rels.append({'meeting_id': meeting_id, 'city_name': city_name})

        for item_id, item_data in item_info.items():
            item_type = item_data.get('type', 'Unknown')
            all_item_type_data.append({'type_name': item_type})
            all_item_data.append({
                'item_id':    item_id,
                'start_time': item_data.get('startTime', ''),
                'end_time':   item_data.get('endTime', ''),
                'duration':   item_data.get('duration', 0),
                'summary':    item_data.get('Summary', '')
            })
            all_item_item_type_rels.append({'item_id': item_id, 'type_name': item_type})
            all_meeting_item_rels.append({'meeting_id': meeting_id, 'item_id': item_id})

            transcript = item_data.get('transcripts', '')
            if transcript:
                for idx, chunk in enumerate(text_splitter.split_text(transcript)):
                    raw_chunks.append({'item_id': item_id, 'text': chunk, 'chunk_index': idx})

    print(f"Collected: {len(all_meeting_data)} meetings, {len(all_item_data)} items, {len(raw_chunks)} chunks")

    # ── 2. Resumable embedding with Drive checkpoints ────────────────────────────
    # Progress file tracks how many chunks have been embedded so far
    progress_file   = os.path.join(drive_cache_dir, "embed_progress.json")
    checkpoint_file = os.path.join(drive_cache_dir, "embeddings_checkpoint.pkl")

    # Load existing progress
    if os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            progress = json.load(f)
        completed_count = progress.get("completed_count", 0)
        print(f"Resuming: {completed_count}/{len(raw_chunks)} chunks already embedded.")
    else:
        completed_count = 0

    # Load existing embedded chunks
    if os.path.exists(checkpoint_file) and completed_count > 0:
        with open(checkpoint_file, 'rb') as f:
            all_transcript_chunk_data = pickle.load(f)
        print(f"Loaded {len(all_transcript_chunk_data)} embedded chunks from Drive.")
    else:
        all_transcript_chunk_data = []

    # Embed remaining chunks in mini-batches, saving to Drive after each batch
    remaining = raw_chunks[completed_count:]
    if remaining:
        print(f"Generating embeddings for {len(remaining)} remaining chunks (batch size={embed_batch_size})...")
        for batch_start in range(0, len(remaining), embed_batch_size):
            batch_chunks = remaining[batch_start: batch_start + embed_batch_size]
            texts        = [c['text'] for c in batch_chunks]
            embeddings   = embedding_model.embed_documents(texts)

            for i, chunk_info in enumerate(batch_chunks):
                all_transcript_chunk_data.append({
                    'item_id':     chunk_info['item_id'],
                    'text':        chunk_info['text'],
                    'embedding':   embeddings[i],
                    'chunk_index': chunk_info['chunk_index']
                })

            # Checkpoint to Drive
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(all_transcript_chunk_data, f)
            new_count = completed_count + batch_start + len(batch_chunks)
            with open(progress_file, 'w') as f:
                json.dump({"completed_count": new_count}, f)
            print(f"  Checkpointed: {new_count}/{len(raw_chunks)} chunks embedded ✓")

    print(f"\nEmbedding complete. Total chunks with embeddings: {len(all_transcript_chunk_data)}")

    # ── 3. Ingest into Neo4j (MERGE = idempotent, safe to re-run) ───────────────
    with driver.session(database=database) as session:

        print("Ingesting City nodes...")
        for i in range(0, len(all_city_data), batch_size):
            session.run("UNWIND $b AS row MERGE (c:City {city_name: row.city_name})",
                        b=all_city_data[i:i+batch_size])
        print(f"  Done: {len(set(d['city_name'] for d in all_city_data))} unique cities")

        print("Ingesting Meeting nodes...")
        for i in range(0, len(all_meeting_data), batch_size):
            session.run("""
                UNWIND $b AS row
                MERGE (m:Meeting {meeting_id: row.meeting_id})
                ON CREATE SET m.webpage_link = row.webpage,
                              m.video_link = row.video,
                              m.meetingdetail_link = row.meetingdetail,
                              m.video_duration = row.videoduration,
                              m.meeting_date = row.meeting_date
            """, b=all_meeting_data[i:i+batch_size])
        print(f"  Done: {len(all_meeting_data)} meetings")

        print("Ingesting Meeting-City relationships...")
        for i in range(0, len(all_meeting_city_rels), batch_size):
            session.run("""
                UNWIND $b AS row
                MATCH (m:Meeting {meeting_id: row.meeting_id})
                MATCH (c:City {city_name: row.city_name})
                MERGE (m)-[:TAKES_PLACE_IN]->(c)
            """, b=all_meeting_city_rels[i:i+batch_size])
        print(f"  Done: {len(all_meeting_city_rels)} relationships")

        print("Ingesting ItemType nodes...")
        for i in range(0, len(all_item_type_data), batch_size):
            session.run("UNWIND $b AS row MERGE (it:ItemType {type_name: row.type_name})",
                        b=all_item_type_data[i:i+batch_size])
        print(f"  Done: {len(set(d['type_name'] for d in all_item_type_data))} unique types")

        print("Ingesting Item nodes...")
        for i in range(0, len(all_item_data), batch_size):
            session.run("""
                UNWIND $b AS row
                MERGE (i:Item {item_id: row.item_id})
                ON CREATE SET i.start_time = row.start_time,
                              i.end_time = row.end_time,
                              i.duration = row.duration,
                              i.summary = row.summary
            """, b=all_item_data[i:i+batch_size])
        print(f"  Done: {len(all_item_data)} items")

        print("Ingesting Item-ItemType relationships...")
        for i in range(0, len(all_item_item_type_rels), batch_size):
            session.run("""
                UNWIND $b AS row
                MATCH (i:Item {item_id: row.item_id})
                MATCH (it:ItemType {type_name: row.type_name})
                MERGE (i)-[:OF_TYPE]->(it)
            """, b=all_item_item_type_rels[i:i+batch_size])
        print(f"  Done: {len(all_item_item_type_rels)} relationships")

        print("Ingesting Meeting-Item relationships...")
        for i in range(0, len(all_meeting_item_rels), batch_size):
            session.run("""
                UNWIND $b AS row
                MATCH (m:Meeting {meeting_id: row.meeting_id})
                MATCH (i:Item {item_id: row.item_id})
                MERGE (m)-[:HAS_ITEM]->(i)
            """, b=all_meeting_item_rels[i:i+batch_size])
        print(f"  Done: {len(all_meeting_item_rels)} relationships")

        print("Ingesting TranscriptChunk nodes and HAS_CHUNK relationships...")
        for i in range(0, len(all_transcript_chunk_data), batch_size):
            session.run("""
                UNWIND $b AS row
                MERGE (c:TranscriptChunk {text: row.text, chunk_index: row.chunk_index})
                ON CREATE SET c.embedding = row.embedding
                WITH c, row
                MATCH (item:Item {item_id: row.item_id})
                MERGE (item)-[:HAS_CHUNK]->(c)
            """, b=all_transcript_chunk_data[i:i+batch_size])
            print(f"  Chunk batch {i//batch_size + 1} ingested ({min(i+batch_size, len(all_transcript_chunk_data))}/{len(all_transcript_chunk_data)})")

    print("\n✅ All data successfully ingested into Neo4j!")
    print(f"Cache files in: {drive_cache_dir}")
    print("To start fresh next time, delete the cache files from Drive.")

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1.5",
    model_kwargs={"trust_remote_code": True}
)

# text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=20)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

In [ ]:
import json
driver = get_neo4j_driver()
setup_neo4j_constraints(driver)
create_vector_index(driver, embeddings)


with open('preprocessed_MeetingBank.json', 'r', encoding='utf-8') as f:
  data = json.load(f)

ingest_data_resumable(driver, data, embeddings, text_splitter)
driver.close()

In [ ]:
driver = get_neo4j_driver()
with driver.session(database = os.getenv("NEO4J_DATABASE")) as session:
    result = session.run("MATCH (c:TranscriptChunk) RETURN count(c) AS totalChunkNodes")
    for record in result:
        print(f"Total TranscriptChunk nodes in Neo4j: {record['totalChunkNodes']}")
driver.close()

In [ ]:
driver = get_neo4j_driver()
with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
    result = session.run("MATCH (c:City) RETURN DISTINCT c.city_name AS city_name")
    city_names = [record["city_name"] for record in result]
    print("Unique City Names:")
    for city in city_names:
        print(f"- {city}")
driver.close()

In [ ]:
driver = get_neo4j_driver()
with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
    result = session.run("MATCH (it:ItemType) RETURN DISTINCT it.type_name AS type_name")
    item_type_names = [record["type_name"] for record in result]
    print("Found : ", len(item_type_names), "unique item type names")
    for item_type in item_type_names:
        print(f"- {item_type}")
driver.close()

# Retrieval Pipeline

**STAGE 1: VECTOR SEARCH**
Finds top-k TranscriptChunks semantically similar to the query.
Returns item_id so Stage 2 can traverse up the graph.

In [ ]:
stage_1_semantic_query = """
  CALL db.index.vector.queryNodes('transcript_chunk_index',$top_k,$query_embedding)
  YIELD node AS chunk,score
  MATCH (chunk)<-[:HAS_CHUNK]-(item:Item)
  RETURN item.item_id AS itemId,
  score,
  chunk.text AS evidence
  ORDER BY score DESC
"""

def semantic_search(driver, query_embedding, top_k = 20):
  """
    Stage 1: Pure vector similarity search on TranscriptChunk nodes.
    Returns a list of {itemId, score, evidence} dicts.
  """
  with driver.session(database = os.getenv("NEO4J_DATABASE")) as session:
    result = session.run(
        stage_1_semantic_query,
        query_embedding = query_embedding,
        top_k = top_k,
      )
    return [record.data() for record in result]


**STAGE 2: GRAPH TRAVERSAL — FILTER + AUGMENT**
Filter = item_type and city are the fields which can be used for filtering
Augment= meeting properties, item properties details can be used to augment the context. basically everything else

In [ ]:
stage_2_filter_and_augment_query = """
  //1. unpack item data from stage 2
  UNWIND $item_data AS data

  //2. Match the item node
  MATCH (item:Item{item_id: data.itemId})

  //3. Traverse up to Meeting and City
  MATCH (meeting:Meeting)-[:HAS_ITEM]->(item)
  MATCH (meeting:Meeting)-[:TAKES_PLACE_IN]->(city:City)

  //4. Filter by city - pass null to skip
  WHERE $city_name IS NULL OR city.city_name = $city_name

  //5. Traverse to ItemType
  MATCH (item)-[:OF_TYPE]->(itemType:ItemType)

  //6. Filter by itemType - pass null to skip
  WITH item, meeting, city, itemType, data
  WHERE $type_name IS NULL OR itemType.type_name = $type_name

  //7. Augment- return everything useful for the LLM
  RETURN
    item.item_id            AS itemId,
    data.score              AS score,
    data.evidence           AS evidence,
    item.summary            AS itemSummary,
    item.start_time         AS startTime,
    item.end_time           AS endTime,
    item.duration           AS duration,
    itemType.type_name         AS itemType,
    city.city_name          AS city,
    meeting.meeting_id      AS meetingId,
    meeting.webpage_link    AS webpageLink,
    meeting.video_link      AS videoLink,
    meeting.meetingdetail_link AS meetingDetailLink,
    meeting.meeting_date       AS meetingDate
  ORDER BY score DESC
"""

def graph_filter_and_augment(driver, semantic_results, city_name = None, type_name = None):
  with driver.session(database = os.getenv("NEO4J_DATABASE")) as session:
    result = session.run(
      stage_2_filter_and_augment_query,
      item_data = semantic_results,
      city_name = city_name,
      type_name = type_name,
    )
    return [record.data() for record in result]

# Augmentation
Build context string for LLM from the "Retrieval" stage results


In [ ]:
def build_context(enriched_results):
  """
  Converts enriched Stage 2 results into a string for the LLM prompt.
  """
  if not enriched_results:
    return "No relevant meeting items found"

  parts = []

  for r in enriched_results:
    parts.append(
        f"[City: {r['city']} | Meeting: {r['meetingId']} | "
        f"Type: {r.get('itemType', 'Unknown')} | Score: {r['score']:.4f}]\n"
        f"Evidence: {r['evidence']}\n"
        f"Item Summary: {r.get('itemSummary', 'N/A')}"
    )

  return "\n\n---\n\n".join(parts)

# Generation
Pass user query and context string from "Augmentation" stage to LLM to generate answers

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                You are an expert assistant analyzing U.S. city council meeting records
                from LongBeach, Seattle, Denver, KingCounty, Alameda, Boston.
                Your task is to answer user questions based the council meetings and the child items information
                - Use only the provided context. Do not make up information.
                - Do NOT add geographic, demographic, or any other details not stated in the context
                - Present the result in a clear, professional format.
                - If the context doesn't contain enough information, say so explicitly.
                - Do NOT render markdown — no ## headers, no ** bold**, no bullet dashes. Write in plain prose only
                - Be concise but complete.
                """
            ),
            (
                "human",
                """
                User Query:
                {user_query}

                Retrieved Context:
                {context}

                Please provide your analysis.
                """
            ),
        ]
    )

def generate_answer(llm, user_query, context):
  chain = RAG_PROMPT | llm
  response = chain.invoke({"user_query": user_query, "context":context})
  return response.content

# Full Graph RAG Pipeline

Entity Extraction for filters - city_name and type_name

In [ ]:
ENTITY_EXTRACTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a query parser for a U.S. city council meeting search system.
Extract structured filters from the user's question.
city_name: One of ["LongBeach","Seattle","Denver","KingCounty","Alameda","Boston"] or null.
  Normalize variants: "Long Beach" → "LongBeach", case-insensitive.
  Only set if the user explicitly mentions a city.
  If the question starts with "which city", "what city", or is asking you to identify the city — return null.
type_name: One of [
    "Agenda Item", "Public Hearing", "Ordinance", "Resolution", "Contract",
    "Emergency Ordinance", "Appointment", "ABC License", "Ordinance (Ord)",
    "Clerk File (CF)", "Resolution (Res)", "Council Bill (CB)",
    "Council Budget Action (CBA)", "Bill", "Proclamation", "Communication",
    "Executive Session", "Presentation", "Announcement", "Motion",
    "Consent Calendar Item", "Continued Agenda Item", "Regular Agenda Item",
    "Joint Agenda Item", "Council Communication", "Council Referral",
    "Joint Consent Item", "Proclamation/Special Order", "SACIC Consent Item",
    "Closed Session Item", "SACIC Regular Item", "Mayor Order",
    "Report of Public Officer", "Committee Reports", "Council Ordinance",
    "Council Hearing Order", "Council Legislative Resolution",
    "Personnel Orders", "Matters Recently Heard-For Possible Action",
    "Mayor Home Rule Petition", "Council Home Rule Petition", "Loan Order",
    "Council 17F Order", "Mayor Ordinance", "Council Order"
] or null.
  Only set if the user explicitly refers to a meeting item type.
Respond ONLY with valid JSON. No explanation, no markdown.
Example: {{"city_name": null, "type_name": null}}
Never return a list. If multiple or all cities apply, return null."""),
    ("human", "User question: {user_query}")

])

In [ ]:
def extract_query_filters(llm, user_query: str) -> dict:
    """
    Uses the LLM to extract city_name and type_name from a natural language query.
    Falls back to None, None on any failure so the pipeline still runs.
    """
    chain = ENTITY_EXTRACTION_PROMPT | llm | JsonOutputParser()
    try:
        filters = chain.invoke({"user_query": user_query})
        city  = filters.get("city_name") or None
        tname = filters.get("type_name") or None
        print(f"[Extracted filters] city_name={city!r}, type_name={tname!r}")
        return {"city_name": city, "type_name": tname}
    except Exception as e:
        print(f"[Entity extraction failed: {e}] — running without filters.")
        return {"city_name": None, "type_name": None}

In [ ]:
# top_k=20: covers ~6 cities with headroom;
# 20/133k = 0.015% of corpus, balancing recall vs. context noise
def graph_rag_query(driver, embeddings, smart_llm, user_query, city_name = None, type_name = None, top_k = 20):
  """
  Full Graph RAG pipeline.
  city_name / type_name can be passed manually for testing or left as None to trigger automatic extraction from the query.
  """

  # Auto-extract filters from the query
  if city_name is None and type_name is None:
    filters = extract_query_filters(fast_llm, user_query)
    city_name = filters["city_name"]
    type_name = filters["type_name"]

  query_embedding = embeddings.embed_query(user_query)
  semantic_results = semantic_search(driver, query_embedding, top_k=top_k)

  if not semantic_results:
    return {"answer": "No relevant chunks found", "enriched_results":[],"context":""}

  enriched_results = graph_filter_and_augment(driver, semantic_results,city_name=city_name,type_name=type_name)
  context = build_context(enriched_results)
  answer = generate_answer(smart_llm, user_query, context)

  return {"answer": answer, "enriched_results":enriched_results,"context": context}

**MAIN**

In [ ]:
driver = get_neo4j_driver()
smart_llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)
fast_llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

In [ ]:
# user_query = "When is the Hispanic Heritage Month in the city of Boston?"
# user_query = "In some city meeting the Hispanic Month was recognized, what is that city name?"
user_query = "What housing affordability measures were discussed in Denver?"

print(graph_rag_query(driver, embeddings, smart_llm,user_query=user_query))

# Streamlit app

In [ ]:
!pip install -q streamlit
!pip install langchain-groq

In [ ]:
%%writefile app.py
"""
MeetingBank GraphRAG — Streamlit Demo (Colab version)
Run with: streamlit run app.py
"""

import os
import streamlit as st
from neo4j import GraphDatabase
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# ── Credentials (Colab testing only — never push this to GitHub) ──────────────
NEO4J_URI      = os.environ.get("NEO4J_URI")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE")
GROQ_KEY = os.environ.get("GROQ_KEY")

# ── Page config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="MeetingBank GraphRAG",
    page_icon="🏛️",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Custom CSS ────────────────────────────────────────────────────────────────
st.markdown("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Mono:wght@400;500&family=DM+Sans:wght@300;400;500&display=swap');

  :root {
    --ink:     #0f1117;
    --paper:   #f5f0e8;
    --accent:  #b5451b;
    --accent2: #2563a8;
    --rule:    #d4c9b0;
    --muted:   #6b6050;
    --card-bg: #faf7f2;
  }

  html, body, [class*="css"] {
    font-family: 'DM Sans', sans-serif;
    background-color: var(--paper) !important;
    color: var(--ink);
  }

  section[data-testid="stSidebar"] {
    background-color: var(--ink) !important;
    border-right: 3px solid var(--accent);
  }
  section[data-testid="stSidebar"] * { color: var(--paper) !important; }
  section[data-testid="stSidebar"] .stSelectbox label {
    color: var(--rule) !important;
    font-size: 0.78rem;
    letter-spacing: 0.08em;
    text-transform: uppercase;
  }

  h1 { font-family: 'DM Serif Display', serif; font-size: 2.6rem !important; color: var(--ink) !important; }
  h2 { font-family: 'DM Serif Display', serif; color: var(--ink) !important; }

  .result-card {
    background: var(--card-bg);
    border: 1px solid var(--rule);
    border-left: 4px solid var(--accent);
    border-radius: 2px;
    padding: 1.2rem 1.4rem;
    margin-bottom: 1rem;
  }
  .result-card:hover { border-left-color: var(--accent2); }

  .card-meta {
    font-family: 'DM Mono', monospace;
    font-size: 0.72rem;
    color: var(--muted);
    margin-bottom: 0.5rem;
    display: flex;
    gap: 1rem;
    flex-wrap: wrap;
  }
  .card-meta span { background: var(--rule); padding: 2px 8px; border-radius: 2px; }
  .card-meta .city  { background: #dce8f5; color: var(--accent2); }
  .card-meta .score { background: #f5dcd4; color: var(--accent); }
  .card-meta .type  { background: #e8f0e0; color: #2d5a1b; }

  .evidence-text {
    font-size: 0.88rem;
    color: var(--muted);
    border-left: 2px solid var(--rule);
    padding-left: 0.8rem;
    margin: 0.6rem 0;
    font-style: italic;
    line-height: 1.6;
  }
  .summary-text { font-size: 0.92rem; color: var(--ink); line-height: 1.7; }

  .answer-box {
    background: var(--ink);
    color: var(--paper);
    border-radius: 2px;
    padding: 1.6rem 2rem;
    font-size: 0.97rem;
    line-height: 1.8;
    border-left: 5px solid var(--accent);
    margin-bottom: 1.5rem;
  }

  .stTextArea > div > div > textarea {
    background: white !important;
    border: 2px solid var(--rule) !important;
    border-radius: 2px !important;
    font-family: 'DM Sans', sans-serif !important;
    color: var(--ink) !important;
  }
  .stTextArea > div > div > textarea:focus {
    border-color: var(--accent) !important;
    box-shadow: none !important;
  }

  .stButton > button {
    background: var(--accent) !important;
    color: white !important;
    border: none !important;
    border-radius: 2px !important;
    font-family: 'DM Mono', monospace !important;
    font-size: 0.8rem !important;
    letter-spacing: 0.1em !important;
    text-transform: uppercase !important;
    padding: 0.6rem 2rem !important;
  }
  .stButton > button:hover { background: var(--accent2) !important; }

  .divider { border: none; border-top: 1px solid var(--rule); margin: 1.5rem 0; }
  a { color: var(--accent2) !important; }

  .stage-label {
    font-family: 'DM Mono', monospace;
    font-size: 0.7rem;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    color: var(--muted);
    margin-bottom: 0.3rem;
  }
</style>
""", unsafe_allow_html=True)


# ── Load resources ────────────────────────────────────────────────────────────

@st.cache_resource(show_spinner=False)
def load_resources():
    os.environ["NEO4J_DATABASE"] = NEO4J_DATABASE

    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
    )
    embeddings = HuggingFaceEmbeddings(
        model_name="nomic-ai/nomic-embed-text-v1.5",
        model_kwargs={"trust_remote_code": True}
    )

    # llama-3.1-8b-instant for fast filter extraction
    fast_llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature = 0,
        api_key=GROQ_KEY)


    #meta-llama/llama-4-scout-17b-16e-instruct for answer generation
    smart_llm = ChatGroq(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        temperature = 0,
        api_key = GROQ_KEY)

    return driver, embeddings, fast_llm, smart_llm


# ── Pipeline functions (mirrors notebook exactly) ─────────────────────────────

STAGE_1_QUERY = """
  CALL db.index.vector.queryNodes('transcript_chunk_index', $top_k, $query_embedding)
  YIELD node AS chunk, score
  MATCH (chunk)<-[:HAS_CHUNK]-(item:Item)
  RETURN item.item_id AS itemId,
         score,
         chunk.text AS evidence
  ORDER BY score DESC
"""

STAGE_2_QUERY = """
  UNWIND $item_data AS data
  MATCH (item:Item {item_id: data.itemId})
  MATCH (meeting:Meeting)-[:HAS_ITEM]->(item)
  MATCH (meeting:Meeting)-[:TAKES_PLACE_IN]->(city:City)
  WHERE $city_name IS NULL OR city.city_name = $city_name
  MATCH (item)-[:OF_TYPE]->(itemType:ItemType)
  WITH item, meeting, city, itemType, data
  WHERE $type_name IS NULL OR itemType.type_name = $type_name
  RETURN
    item.item_id               AS itemId,
    data.score                 AS score,
    data.evidence              AS evidence,
    item.summary               AS itemSummary,
    item.start_time            AS startTime,
    item.end_time              AS endTime,
    item.duration              AS duration,
    itemType.type_name         AS itemType,
    city.city_name             AS city,
    meeting.meeting_id         AS meetingId,
    meeting.webpage_link       AS webpageLink,
    meeting.video_link         AS videoLink,
    meeting.meetingdetail_link AS meetingDetailLink,
    meeting.meeting_date       AS meetingDate
  ORDER BY score DESC
"""
ENTITY_EXTRACTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a query parser for a U.S. city council meeting search system.
Extract structured filters from the user's question.
city_name: One of ["LongBeach","Seattle","Denver","KingCounty","Alameda","Boston"] or null.
  Normalize variants: "Long Beach" → "LongBeach", case-insensitive.
  Only set if the user explicitly mentions a city.
  If the question starts with "which city", "what city", or is asking you to identify the city — return null.
type_name: One of [
    "Agenda Item", "Public Hearing", "Ordinance", "Resolution", "Contract",
    "Emergency Ordinance", "Appointment", "ABC License", "Ordinance (Ord)",
    "Clerk File (CF)", "Resolution (Res)", "Council Bill (CB)",
    "Council Budget Action (CBA)", "Bill", "Proclamation", "Communication",
    "Executive Session", "Presentation", "Announcement", "Motion",
    "Consent Calendar Item", "Continued Agenda Item", "Regular Agenda Item",
    "Joint Agenda Item", "Council Communication", "Council Referral",
    "Joint Consent Item", "Proclamation/Special Order", "SACIC Consent Item",
    "Closed Session Item", "SACIC Regular Item", "Mayor Order",
    "Report of Public Officer", "Committee Reports", "Council Ordinance",
    "Council Hearing Order", "Council Legislative Resolution",
    "Personnel Orders", "Matters Recently Heard-For Possible Action",
    "Mayor Home Rule Petition", "Council Home Rule Petition", "Loan Order",
    "Council 17F Order", "Mayor Ordinance", "Council Order"
] or null.
  Only set if the user explicitly refers to a meeting item type.
Respond ONLY with valid JSON. No explanation, no markdown.
Example: {{"city_name": null, "type_name": null}}
Never return a list. If multiple or all cities apply, return null."""),
    ("human", "User question: {user_query}")

])

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are an expert assistant analyzing U.S. city council meeting records
from Seattle, King County, Denver, Boston, Alameda, and Long Beach.
Your task is to answer user questions based the council meetings and the child items information
- Use only the provided context. Do not make up information.
- Do NOT add geographic, demographic, or any other details not stated in the context
- Present results in a clear, professional format.
- If the context doesn't contain enough information, say so explicitly.
- - Do NOT render markdown — no ## headers, no ** bold**, no bullet dashes. Write in plain prose only
- Be concise but complete."""),
    ("human", "User Query:\n{user_query}\n\nRetrieved Context:\n{context}\n\nPlease provide your analysis.")
])


def semantic_search(driver, query_embedding, top_k=20):
    db = os.getenv("NEO4J_DATABASE")
    with driver.session(database=db) as session:
        result = session.run(STAGE_1_QUERY,
                             query_embedding=query_embedding, top_k=top_k)
        return [record.data() for record in result]


def graph_filter_and_augment(driver, semantic_results, city_name=None, type_name=None):
    db = os.getenv("NEO4J_DATABASE")
    with driver.session(database=db) as session:
        result = session.run(STAGE_2_QUERY,
                             item_data=semantic_results,
                             city_name=city_name,
                             type_name=type_name)
        return [record.data() for record in result]


def build_context(enriched_results):
    if not enriched_results:
        return "No relevant meeting items found"
    parts = []
    for r in enriched_results:
        parts.append(
            f"[City: {r['city']} | Meeting: {r['meetingId']} | "
            f"Type: {r.get('itemType', 'Unknown')} | Score: {r['score']:.4f}]\n"
            f"Evidence: {r['evidence']}\n"
            f"Item Summary: {r.get('itemSummary', 'N/A')}"
        )
    return "\n\n---\n\n".join(parts)


def extract_query_filters(fast_llm, user_query):
    chain = ENTITY_EXTRACTION_PROMPT | fast_llm | JsonOutputParser()
    try:
        filters = chain.invoke({"user_query": user_query})
        return {
            "city_name": filters.get("city_name") or None,
            "type_name": filters.get("type_name") or None,
        }
    except Exception:
        return {"city_name": None, "type_name": None}


def generate_answer(smart_llm, user_query, context):
    chain = RAG_PROMPT | smart_llm
    response = chain.invoke({"user_query": user_query, "context": context})
    return response.content


# ── UI ────────────────────────────────────────────────────────────────────────

st.markdown("""
<div style="border-bottom: 3px solid #0f1117; padding-bottom: 1rem; margin-bottom: 1.5rem;">
  <div style="font-family:'DM Mono',monospace; font-size:0.72rem; letter-spacing:0.15em;
              color:#b5451b; text-transform:uppercase; margin-bottom:0.2rem;">
    Portfolio Project · Graph RAG
  </div>
  <h1 style="margin:0; line-height:1.1;">MeetingBank<br>
    <span style="color:#b5451b;">Council Intelligence</span>
  </h1>
  <div style="font-family:'DM Sans',sans-serif; color:#6b6050; margin-top:0.5rem; font-size:0.95rem;">
    1,250 meetings · 6,894 agenda items · 133,536 transcript chunks across 6 U.S. cities
  </div>
</div>
""", unsafe_allow_html=True)

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("""
    <div style="font-family:'DM Serif Display',serif; font-size:1.4rem; margin-bottom:0.2rem;">
      Search Filters
    </div>
    <div style="font-size:0.75rem; color:#9a8f80; margin-bottom:1.2rem;">
      Optionally narrow results by city or item type.<br>
      Leave on Auto-detect to let the model decide.
    </div>
    """, unsafe_allow_html=True)

    city_filter = st.selectbox(
        "City",
        ["Auto-detect", "LongBeach","Seattle","Denver","KingCounty","Alameda","Boston"]
    )
    type_filter = st.selectbox(
        "Item Type",
        ["Auto-detect", "Agenda Item", "Committee Reports", "Proclamation",
         "Proclamation/Special Order", "Public Comment", "Consent Calendar"]
    )
    top_k = st.slider("Top-K chunks (Stage 1)", min_value=5, max_value=50,
                      value=20, step=5)

    st.markdown("<hr style='border-color:#2a2a2a; margin:1rem 0;'>",
                unsafe_allow_html=True)
    st.markdown("""
    <div style="font-size:0.7rem; color:#6b5f50; line-height:1.7;">
      <b style="color:#d4c9b0;">Two-Stage Graph RAG</b><br>
      ① Vector search on TranscriptChunks<br>
      ② Graph traversal for metadata &amp; filtering<br>
    </div>
    """, unsafe_allow_html=True)

# ── Sample queries ────────────────────────────────────────────────────────────
st.markdown("<div class='stage-label'>Try a sample query</div>",
            unsafe_allow_html=True)

samples = [
    "What housing affordability measures were discussed in Denver?",
    "Which city recognized Hispanic Heritage Month and what was said?",
    "What public comments were made about homelessness in Long Beach?",
    "Summarize proclamations made in Boston city council meetings.",
    "What infrastructure or transportation projects came up in Alameda?",
]

# Initialize session state
if "user_query" not in st.session_state:
    st.session_state["user_query"] = ""

cols = st.columns(len(samples))
chosen_sample = None
for i, (col, q) in enumerate(zip(cols, samples)):
    with col:
        if st.button(q[:40] + "…", key=f"sample_{i}", use_container_width=True):
            # chosen_sample = q
            st.session_state["user_query"] = q

# ── Query input ───────────────────────────────────────────────────────────────
st.markdown("<hr class='divider'>", unsafe_allow_html=True)

user_query = st.text_area(
    "Your question",
    # value=chosen_sample or "",
    placeholder="Ask anything about U.S. city council meetings…",
    height=90,
    label_visibility="collapsed",
    key="user_query"
)

# Keep session state in sync with manual edits
# st.session_state["user_query"] = user_query

run = st.button("⬡  Search Knowledge Graph")

# ── Execute with step-by-step progress ───────────────────────────────────────
if run:
    if not user_query.strip():
        st.warning("Please enter a question.")
        st.stop()

    city_arg = None if city_filter == "Auto-detect" else city_filter
    type_arg = None if type_filter == "Auto-detect" else type_filter

    # Step 0: load resources
    with st.spinner("Connecting to Neo4j and loading models…"):
        try:
            driver, embeddings, fast_llm, smart_llm = load_resources()
        except Exception as e:
            st.error(f"Failed to load resources: {e}")
            st.stop()
    st.success("✅ Connected to Neo4j and loaded models.")

    # Step 1: extract filters (Haiku — fast)
    with st.spinner("Extracting filters from query…"):
        try:
            if city_arg is None and type_arg is None:
                filters = extract_query_filters(fast_llm, user_query)
                city_arg = filters["city_name"]
                type_arg = filters["type_name"]
        except Exception as e:
            st.error(f"Filter extraction failed: {e}")
            st.stop()
    st.success(f"✅ Filters — city: {city_arg or 'none'}, type: {type_arg or 'none'}")

    # Step 2: embed query
    with st.spinner("Generating query embedding…"):
        try:
            query_embedding = embeddings.embed_query(user_query)
        except Exception as e:
            st.error(f"Embedding failed: {e}")
            st.stop()
    st.success(f"✅ Embedding done — vector length: {len(query_embedding)}")

    # Step 3: vector search
    with st.spinner(f"Stage 1: vector search (top_k={top_k})…"):
        try:
            semantic_results = semantic_search(driver, query_embedding, top_k=top_k)
        except Exception as e:
            st.error(f"Vector search failed: {e}")
            st.stop()
    st.success(f"✅ Stage 1 done — {len(semantic_results)} chunks retrieved.")

    if not semantic_results:
        st.warning("No relevant chunks found for this query.")
        st.stop()

    # Step 4: graph traversal
    with st.spinner("Stage 2: graph traversal and enrichment…"):
        try:
            enriched_results = graph_filter_and_augment(
                driver, semantic_results,
                city_name=city_arg, type_name=type_arg
            )
        except Exception as e:
            st.error(f"Graph traversal failed: {e}")
            st.stop()
    st.success(f"✅ Stage 2 done — {len(enriched_results)} rows returned (items + graph context).")

    # Step 5: generate answer
    with st.spinner("Stage 3: generating answer with LLM…"):
        try:
            context = build_context(enriched_results)
            answer  = generate_answer(smart_llm, user_query, context)
        except Exception as e:
            st.error(f"Answer generation failed: {e}")
            st.stop()
    st.success("✅ Answer generated.")

    # ── Results ───────────────────────────────────────────────────────────────
    st.markdown("<hr class='divider'>", unsafe_allow_html=True)
    st.markdown("<h2 style='margin-top:1rem;'>Answer</h2>", unsafe_allow_html=True)
    st.markdown(
        f'<div class="answer-box">{answer.replace(chr(10), "<br>")}</div>',
        unsafe_allow_html=True
    )

    # Stats
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Stage 1 chunks", len(semantic_results))
    c2.metric("Stage 2 enriched rows",  len(enriched_results))
    c3.metric("City filter",    city_arg or "None")
    c4.metric("Type filter",    type_arg or "None")

    # Retrieved items
    if enriched_results:
        st.markdown("<hr class='divider'>", unsafe_allow_html=True)
        st.markdown(
            f"<h2>Retrieved Items "
            f"<span style='font-size:1rem; color:#6b6050;'>({len(enriched_results)} results)</span></h2>",
            unsafe_allow_html=True
        )

        for r in enriched_results:
            links = []
            if r.get("webpageLink"):
                links.append(f'<a href="{r["webpageLink"]}" target="_blank">Webpage</a>')
            if r.get("videoLink"):
                links.append(f'<a href="{r["videoLink"]}" target="_blank">Video</a>')
            if r.get("meetingDetailLink"):
                links.append(f'<a href="{r["meetingDetailLink"]}" target="_blank">Detail</a>')
            links_html = " · ".join(links) if links else ""
            date_str   = r.get("meetingDate", "")
            evidence   = r.get("evidence", "")
            summary    = r.get("itemSummary", "")

            st.markdown(f"""
            <div class="result-card">
              <div class="card-meta">
                <span class="city">🏙 {r.get('city','?')}</span>
                <span class="score">⟐ {r['score']:.4f}</span>
                <span class="type">{r.get('itemType','?')}</span>
                <span>{r.get('meetingId','?')}</span>
                {f'<span>📅 {date_str}</span>' if date_str else ''}
              </div>
              <div class="evidence-text">
                "{evidence[:400]}{'…' if len(evidence) > 400 else ''}"
              </div>
              <div class="summary-text">
                {summary or '<em style="color:#9a8f80;">No summary available</em>'}
              </div>
              {f'<div style="margin-top:0.6rem; font-size:0.8rem;">{links_html}</div>'
               if links_html else ''}
            </div>
            """, unsafe_allow_html=True)

    # Raw context
    with st.expander("View raw LLM context"):
        st.code(context, language="text")

In [ ]:
!pip install pyngrok -q

In [ ]:
from pyngrok import ngrok
import subprocess, time

# Sign up free at https://dashboard.ngrok.com and paste your token here
ngrok.set_auth_token("3D3fdtYBnGPDiucZhJGq2VzTU0C_2AqoNKKtaDCbpauCdqzaZ")

# Start Streamlit
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port=8501",
                  "--server.headless=true"])
time.sleep(4)

# Open tunnel
public_url = ngrok.connect(8501)
print(f"✅ App is live at: {public_url}")

In [ ]:
from pyngrok import ngrok

# Kill ALL running tunnels and the ngrok process
ngrok.kill()

# Verify they're gone
print("Active tunnels:", ngrok.get_tunnels())

# Push to Github

In [ ]:
# Cell 1 — Configure git (run once)
!git config --global user.email "gayatrivlp04263@gmail.com"
!git config --global user.name "gayatrivlp"

In [ ]:
# Cell 2 — Clone your repo into Colab
!git clone https://github.com/gayatrivlp/meetingbank-graph-rag.git
%cd meetingbank-graph-rag

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
notebook_source_path = '/content/drive/MyDrive/Colab Notebooks/Graph_RAG_Project.ipynb' # Example path

import shutil
import os

# Ensure the target directory exists and we are in it
# repo_path = 'meetingbank-graph-rag'
# if not os.path.exists(repo_path):
#     print(f"Cloning {repo_path} first...")
#     !git clone https://github.com/gayatrivlp/meetingbank-graph-rag.git

# %cd {repo_path}

try:
    shutil.copy(notebook_source_path, '.')
    print(f"Successfully copied '{notebook_source_path}' to '{os.getcwd()}'")
except FileNotFoundError:
    print(f"Error: Notebook not found at '{notebook_source_path}'. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred: {e}")

# Go back to the root content directory after copying
%cd /content/


In [ ]:
# Cell 4 — Add, commit, push
!git add .
!git commit -m "Add Graph RAG notebook and Streamlit app"
!git push origin main